In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U qwen-asr

In [ ]:
#!pip install -U qwen-asr[vllm]

In [ ]:
#!pip install -U flash-attn --no-build-isolation

In [ ]:
#!/usr/bin/env python3
import argparse
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from tqdm import tqdm
from dataclasses import dataclass
from typing import Iterable, List, Optional, Sequence, Tuple

COLAB_DEFAULT_INPUT = "/content/drive/MyDrive/Qwen3-ASR-src" # 音频文件所处目录

SUPPORTED_EXTS = {".mp4", ".mp3", ".wmv", ".wav", ".mkv", ".m4a", ".flac", ".aac", ".avi", ".mov"}

@dataclass
class TimestampToken:
    start: float
    end: float
    text: str

@dataclass
class Cue:
    start: float
    end: float
    text: str

def run(cmd: Sequence[str]) -> None:
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "Command failed: {}\nSTDOUT:\n{}\nSTDERR:\n{}".format(
                " ".join(shlex.quote(c) for c in cmd), proc.stdout, proc.stderr
            )
        )

def check_ffmpeg() -> None:
    for tool in ("ffmpeg", "ffprobe"):
        try:
            run([tool, "-version"])
        except Exception as exc:
            raise RuntimeError(f"Missing {tool}. Please install ffmpeg.") from exc

def maybe_mount_google_drive(input_path: str) -> None:
    drive_root = "/content/drive"
    normalized = os.path.abspath(input_path)
    if not normalized.startswith(drive_root + os.sep):
        return
    if os.path.isdir(os.path.join(drive_root, "MyDrive")):
        return
    try:
        from google.colab import drive
    except Exception as exc:
        raise RuntimeError(
            "Input is under /content/drive, but google.colab is unavailable. "
            "Please run in Colab or pass a local path."
        ) from exc
    drive.mount(drive_root)

def extract_audio(input_path: str, wav_path: str) -> None:
    run(
        [
            "ffmpeg",
            "-y",
            "-i",
            input_path,
            "-ac",
            "1",
            "-ar",
            "16000",
            "-vn",
            wav_path,
        ]
    )

def split_audio_with_vad(wav_path: str, out_dir: str) -> List[dict]:
    """使用 Silero VAD 智能切分音频并保存为独立的wav片段"""
    os.makedirs(out_dir, exist_ok=True)
    import torch
    import torchaudio

    print("正在加载 Silero VAD 模型进行智能静音切片...")
    model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                  model='silero_vad',
                                  force_reload=False,
                                  trust_repo=True)
    (get_speech_timestamps, _, read_audio, _, _) = utils

    wav = read_audio(wav_path, sampling_rate=16000)

    # 获取有效语音的时间戳 (已优化轻柔声音检测)
    speech_timestamps = get_speech_timestamps(
        wav, model, sampling_rate=16000,
        threshold=0.3,             # 降低阈值，捕捉轻声
        min_silence_duration_ms=500,
        min_speech_duration_ms=170 # 降低短促音限制
    )

    segments_info = []
    print(f"检测到 {len(speech_timestamps)} 个语音段，正在导出分片...")
    for idx, ts in enumerate(speech_timestamps):
        start_sample = ts['start']
        end_sample = ts['end']

        # 裁剪并保存单段wav
        chunk = wav[start_sample:end_sample].unsqueeze(0)
        out_path = os.path.join(out_dir, f"seg_{idx:04d}.wav")
        torchaudio.save(out_path, chunk, 16000)

        # 记录每段的绝对开始时间和时长
        segments_info.append({
            "path": out_path,
            "start": float(start_sample / 16000.0),
            "duration": float((end_sample - start_sample) / 16000.0)
        })

    if not segments_info:
        raise RuntimeError("No audio segments were created (No speech detected).")
    return segments_info

def load_model(
    backend: str,
    model_name: str,
    dtype: str,
    device: str,
    max_new_tokens: int,
    max_batch: int,
    use_forced_aligner: bool,
    forced_aligner: Optional[str],
    flash_attn2: bool,
):
    import torch
    from qwen_asr import Qwen3ASRModel

    torch_dtype = getattr(torch, dtype)

    common_kwargs = dict(
        dtype=torch_dtype,
        device_map=device,
        max_inference_batch_size=max_batch,
        max_new_tokens=max_new_tokens,
    )
    if flash_attn2:
        can_use_fa2 = torch.cuda.is_available()
        if can_use_fa2:
            major, _ = torch.cuda.get_device_capability(0)
            if major < 8:
                can_use_fa2 = False
                print("Current GPU compute capability < 8; disable FlashAttention 2.")
        if can_use_fa2:
            try:
                import flash_attn  # noqa: F401
                common_kwargs["attn_implementation"] = "flash_attention_2"
            except Exception:
                print("FlashAttention 2 not detected; fallback to default attention.")

    aligner_kwargs = {}
    if use_forced_aligner and forced_aligner:
        aligner_kwargs = dict(
            forced_aligner=forced_aligner,
            forced_aligner_kwargs=dict(
                dtype=torch_dtype,
                device_map=device,
            ),
        )

    def build_transformers_model():
        try:
            return Qwen3ASRModel.from_pretrained(
                model_name,
                **common_kwargs,
                **aligner_kwargs,
            )
        except TypeError:
            if "attn_implementation" not in common_kwargs:
                raise
            common_kwargs.pop("attn_implementation", None)
            print("Current qwen-asr version does not accept attn_implementation; fallback without FlashAttention 2.")
            return Qwen3ASRModel.from_pretrained(
                model_name,
                **common_kwargs,
                **aligner_kwargs,
            )

    if backend == "vllm":
        try:
            return Qwen3ASRModel.LLM(
                model_name,
                dtype=torch_dtype,
                tensor_parallel_size=1,
                max_num_seqs=max_batch,
                gpu_memory_utilization=0.6,
            )
        except Exception as exc:
            print(f"vLLM initialization failed ({exc}); fallback to transformers backend.")
            return build_transformers_model()

    try:
        return build_transformers_model()
    except Exception:
        raise

def normalize_timestamp_token(item) -> Optional[TimestampToken]:
    if item is None:
        return None

    if isinstance(item, dict):
        start = item.get("start")
        end = item.get("end")
        text = item.get("text", "")
        if start is None or end is None:
            return None
        return TimestampToken(float(start), float(end), str(text))

    if isinstance(item, (list, tuple)) and len(item) >= 2:
        start = float(item[0])
        end = float(item[1])
        text = ""
        if len(item) >= 3:
            text = str(item[2])
        return TimestampToken(start, end, text)

    return None

def serialize_time_stamps(items) -> List[dict]:
    serialized: List[dict] = []
    if not items:
        return serialized
    for item in items:
        if isinstance(item, dict):
            start = item.get("start")
            end = item.get("end")
            text = item.get("text", "")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            start = item[0]
            end = item[1]
            text = item[2] if len(item) >= 3 else ""
        else:
            start = getattr(item, "start", None)
            end = getattr(item, "end", None)
            text = getattr(item, "text", "")
        if start is None or end is None:
            continue
        serialized.append({"start": float(start), "end": float(end), "text": str(text)})
    return serialized

def smart_append(buffer: str, piece: str) -> str:
    if not buffer:
        return piece
    if not piece:
        return buffer
    if piece[:1].isspace() or buffer[-1:].isspace():
        return buffer + piece
    if piece[0] in ")]}%.,!?;:" or buffer[-1] in "([{":
        return buffer + piece
    if buffer[-1].isalnum() and piece[0].isalnum():
        return buffer + " " + piece
    return buffer + piece

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def build_cues_from_tokens(tokens: List[TimestampToken], offset: float, max_chars: int, max_duration: float) -> List[Cue]:
    cues: List[Cue] = []
    current_text = ""
    current_start: Optional[float] = None
    current_end: Optional[float] = None

    def should_break(last_piece: str, total_text: str, start: float, end: float) -> bool:
        if re.search(r"[.!?。！？]$", last_piece.strip()):
            return True
        if len(total_text) >= max_chars and (end - start) >= 1.0:
            return True
        if (end - start) >= max_duration:
            return True
        return False

    for token in tokens:
        if not token.text:
            continue
        if current_start is None:
            current_start = token.start
        current_end = token.end
        current_text = smart_append(current_text, token.text)
        if should_break(token.text, current_text, current_start, current_end):
            cues.append(Cue(current_start + offset, current_end + offset, clean_text(current_text)))
            current_text = ""
            current_start = None
            current_end = None

    if current_text and current_start is not None and current_end is not None:
        cues.append(Cue(current_start + offset, current_end + offset, clean_text(current_text)))

    return cues

def fallback_cues_from_text(text: str, offset: float, duration: float, max_chars: int) -> List[Cue]:
    parts = re.split(r"([.!?。！？])", text)
    sentences = []
    buf = ""
    for part in parts:
        if not part:
            continue
        buf += part
        if re.search(r"[.!?。！？]$", part):
            sentences.append(buf)
            buf = ""
    if buf:
        sentences.append(buf)

    lines: List[str] = []
    for sent in sentences:
        sent = clean_text(sent)
        if not sent:
            continue
        while len(sent) > max_chars:
            lines.append(sent[:max_chars])
            sent = sent[max_chars:]
        lines.append(sent)

    if not lines:
        return []

    per = max(duration / len(lines), 0.5)
    cues: List[Cue] = []
    t = 0.0
    for line in lines:
        cues.append(Cue(offset + t, offset + min(t + per, duration), line))
        t += per
    return cues

def format_srt_time(seconds: float) -> str:
    ms = int(round(seconds * 1000.0))
    h = ms // 3600000
    ms -= h * 3600000
    m = ms // 60000
    ms -= m * 60000
    s = ms // 1000
    ms -= s * 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def write_srt(cues: List[Cue], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for i, cue in enumerate(cues, 1):
            f.write(f"{i}\n")
            f.write(f"{format_srt_time(cue.start)} --> {format_srt_time(cue.end)}\n")
            f.write(cue.text + "\n\n")

def derive_output_path(input_path: str) -> str:
    base, _ = os.path.splitext(input_path)
    return base + ".srt"

def derive_tmp_dir(input_path: str) -> str:
    input_dir = os.path.dirname(input_path) or "."
    stem = os.path.splitext(os.path.basename(input_path))[0]
    safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("_") or "input"
    return os.path.join(input_dir, f".tmp_asr_{safe_stem}")

def save_progress(path: str, payload: dict) -> None:
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)

def load_progress(path: str) -> Optional[dict]:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def transcribe_segments(model, segment_paths: List[str], language: Optional[str], return_time_stamps: bool):
    if language is None:
        lang_arg = None
    else:
        lang_arg = [language] * len(segment_paths)

    return model.transcribe(
        audio=segment_paths,
        language=lang_arg,
        return_time_stamps=return_time_stamps,
    )

def get_media_files(directory: str) -> List[str]:
    media_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if os.path.splitext(file)[1].lower() in SUPPORTED_EXTS:
                media_files.append(os.path.join(root, file))
    return sorted(media_files)

def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--input",
        default=COLAB_DEFAULT_INPUT,
        help="Path to a media file OR a directory containing media files.",
    )
    # === 针对 T4 优化的默认参数修改开始 ===
    parser.add_argument("--backend", choices=["transformers", "vllm"], default="transformers")
    parser.add_argument("--model", default="Qwen/Qwen3-ASR-1.7B")
    parser.add_argument("--dtype", default="bfloat16", choices=["float16", "bfloat16"])
    parser.add_argument("--device", default="cuda:0")
    parser.add_argument("--language", default=None, help="Language name or None for auto detection")
    parser.add_argument("--forced-aligner", default="Qwen/Qwen3-ForcedAligner-0.6B", help="Forced aligner model name or local path")
    parser.add_argument("--max-new-tokens", type=int, default=1024)
    parser.add_argument("--max-batch", type=int, default=8)
    # === 针对 T4 优化的默认参数修改结束 ===

    parser.add_argument("--max-chars", type=int, default=42)
    parser.add_argument("--max-duration", type=float, default=6.0)
    parser.add_argument(
        "--flash-attn2",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Enable FlashAttention 2 when backend=transformers (default: enabled)",
    )
    parser.add_argument("--no-timestamps", action="store_true")
    parser.add_argument("--resume", action="store_true", default=True, help="Resume from saved progress (enabled by default in batch mode)")
    parser.add_argument("--retries", type=int, default=2, help="Retry count per batch on failure")
    parser.add_argument("--retry-backoff", type=float, default=5.0, help="Seconds to wait between retries")
    args, unknown = parser.parse_known_args()

    maybe_mount_google_drive(args.input)
    args.input = os.path.abspath(args.input)

    # 扫描收集所有需要处理的文件
    target_files = []
    if os.path.isdir(args.input):
        target_files = get_media_files(args.input)
        if not target_files:
            print(f"在目录中未找到支持的音视频文件: {args.input}")
            return 0
        print(f"找到 {len(target_files)} 个媒体文件，准备进行批量处理...")
    elif os.path.isfile(args.input):
        target_files = [args.input]
    else:
        raise FileNotFoundError(f"Input path not found: {args.input}")

    check_ffmpeg()

    # === 全局只加载一次模型，极大节省批量处理时间 ===
    print("正在加载 Qwen ASR 主模型与对齐模型...")
    model = load_model(
        backend=args.backend,
        model_name=args.model,
        dtype=args.dtype,
        device=args.device,
        max_new_tokens=args.max_new_tokens,
        max_batch=args.max_batch,
        use_forced_aligner=not args.no_timestamps,
        forced_aligner=args.forced_aligner if not args.no_timestamps else None,
        flash_attn2=args.flash_attn2,
    )

    # === 开始遍历处理文件 ===
    for current_file in target_files:
        print(f"\n{'='*50}\n正在处理: {os.path.basename(current_file)}\n{'='*50}")

        current_output = derive_output_path(current_file)
        current_tmp = derive_tmp_dir(current_file)

        # 断点跳过：如果 SRT 已存在，直接跳过处理下一个
        if os.path.exists(current_output):
            print(f"SRT 字幕已存在，跳过此文件: {current_output}")
            continue

        os.makedirs(os.path.dirname(current_output) or ".", exist_ok=True)

        progress_path = os.path.join(current_tmp, "progress.json")
        progress = load_progress(progress_path) if args.resume else None

        segments_dir = os.path.join(current_tmp, "segments")
        wav_path = os.path.join(current_tmp, "audio_16k.wav")

        if progress is None:
            os.makedirs(current_tmp, exist_ok=True)
            print("正在提取并重采样音频...")
            extract_audio(current_file, wav_path)

            if os.path.exists(segments_dir):
                shutil.rmtree(segments_dir)
            os.makedirs(segments_dir, exist_ok=True)

            try:
                segments_info = split_audio_with_vad(wav_path, segments_dir)
            except RuntimeError as e:
                print(f"跳过此文件 ({e})")
                shutil.rmtree(current_tmp, ignore_errors=True)
                continue

            progress = {
                "input": current_file,
                "segments_info": segments_info,
                "no_timestamps": args.no_timestamps,
                "completed": {},
            }
            save_progress(progress_path, progress)
        else:
            if progress.get("input") != current_file:
                # 容错处理
                shutil.rmtree(current_tmp, ignore_errors=True)
                print("检测到历史遗留废弃缓存，已清理，请重新运行本代码块。")
                continue
            if "segments_info" not in progress:
                shutil.rmtree(current_tmp, ignore_errors=True)
                print("发现不兼容的旧版本缓存，已清理，请重新运行本代码块。")
                continue
            segments_info = progress["segments_info"]
            if not segments_info:
                print("恢复失败：未找到有效切片。")
                continue

        segments = [s["path"] for s in segments_info]
        cues: List[Cue] = []
        batch_size = args.max_batch

        for i in tqdm(range(0, len(segments), batch_size), desc="转录进度"):
            batch = segments[i : i + batch_size]
            batch_indices = list(range(i, min(i + batch_size, len(segments))))
            pending = [idx for idx in batch_indices if str(idx) not in progress["completed"]]

            if pending:
                pending_paths = [segments[idx] for idx in pending]
                last_err = None
                for attempt in range(args.retries + 1):
                    try:
                        results = transcribe_segments(
                            model,
                            pending_paths,
                            language=args.language,
                            return_time_stamps=not args.no_timestamps,
                        )
                        last_err = None
                        break
                    except Exception as exc:
                        last_err = exc
                        if attempt < args.retries:
                            time.sleep(args.retry_backoff)
                if last_err is not None:
                    print(f"\n批次推理失败，跳过剩余部分: {last_err}")
                    break

                for idx, result in zip(pending, results):
                    entry = {"text": getattr(result, "text", "")}
                    if not args.no_timestamps and getattr(result, "time_stamps", None):
                        entry["time_stamps"] = serialize_time_stamps(result.time_stamps)
                    progress["completed"][str(idx)] = entry
                save_progress(progress_path, progress)

            for j, seg_path in enumerate(batch):
                seg_info = segments_info[i + j]
                seg_duration = seg_info["duration"]
                offset = seg_info["start"]

                cached = progress["completed"].get(str(i + j))
                if cached:
                    if not args.no_timestamps and cached.get("time_stamps"):
                        tokens = []
                        for item in cached["time_stamps"]:
                            tok = normalize_timestamp_token(item)
                            if tok:
                                tokens.append(tok)
                        cues.extend(
                            build_cues_from_tokens(tokens, offset, args.max_chars, args.max_duration)
                        )
                    else:
                        cues.extend(
                            fallback_cues_from_text(
                                cached.get("text", ""), offset, seg_duration, args.max_chars
                            )
                        )

        write_srt(cues, current_output)
        print(f"完成！SRT 已保存至: {current_output}")

        # 处理完毕后，自动清理当前视频的临时切片，释放硬盘空间
        shutil.rmtree(current_tmp, ignore_errors=True)

    print("\n所有指定目录下的文件已全部处理完毕！")
    return 0

if __name__ == "__main__":
    main()